# DiffuSVG — Dataset Build + Qwen2VL LoRA Fine-Tune

> **Runtime**: GPU (T4 16 GB).  
> Set `HF_TOKEN` in the code cell below, then **Runtime → Run all**.
>
> **Pipeline**: bad-prompts from `results.json`  
> → SD 3.5 Medium → Potrace silhouette SVG  
> → Qwen2VL lenient-silhouette gate → JSONL dataset  
> → QLoRA SFT on Qwen2VL-7B-Instruct


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
#  DiffuSVG — Full Pipeline in One Cell
#  Text Prompt -> SD 3.5 Medium -> Potrace -> Qwen2VL gate -> LoRA fine-tune
# ═══════════════════════════════════════════════════════════════════════════

# ── CONFIG — fill in HF_TOKEN before running ──────────────────────────────
HF_TOKEN        = "YOUR_HF_TOKEN_HERE"
QUIVER_API_KEY  = "YOUR_QUIVER_KEY_HERE"  # app.quiver.ai                          # huggingface.co/settings/tokens
RESULTS_JSON    = "results.json"
OUTPUT_DIR      = "./dataset"
LORA_OUTPUT_DIR = "./qwen2vl_svg_lora"
CLIP_THRESHOLD  = 24.0
DINO_THRESHOLD  = 0.35
SD_STEPS        = 20
SD_GUIDANCE     = 7.5
SD_SEED         = 42
USE_VLM_FILTER  = True    # Qwen2-VL-2B: small enough for CPU filter gate (whiteboard Y/X)
VLM_MODEL       = "Qwen/Qwen2-VL-2B-Instruct"
EPOCHS          = 10
FORCE_REGEN     = True   # delete existing dataset and re-run with VLM filter
BATCH_SIZE      = 1   # T4 16 GB: must be 1 to fit Qwen2-VL-7B 4-bit + LoRA activations
GRAD_ACCUM      = 16  # keeps effective batch = 16
LEARNING_RATE   = 2e-4
LORA_R          = 32
LORA_ALPHA      = 64
assert HF_TOKEN, "Set HF_TOKEN above"

# ── STEP 0: Install ───────────────────────────────────────────────────────
import subprocess, shutil, sys, os, gc, json, logging, base64, io
from pathlib import Path

# Reduce CUDA memory fragmentation
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

def _run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0: print(f"WARN: {r.stderr.strip()[-200:]}")
    return r.returncode == 0

print("apt-get update ...")
_run(["apt-get", "update", "-q"])
for pkg in ["potrace", "imagemagick"]:
    print(f"Installing {pkg} ...")
    _run(["apt-get", "install", "-y", "--fix-missing", "--no-install-recommends", pkg])

missing = [t for t in ["potrace", "convert"] if not shutil.which(t)]
if missing:
    raise EnvironmentError(f"Tools not found: {missing}. Try: Runtime -> Disconnect and delete runtime.")

print("potrace :", subprocess.run(["potrace","--version"], capture_output=True, text=True).stdout.split("\n")[0])
print("convert :", subprocess.run(["convert","--version"],  capture_output=True, text=True).stdout.split("\n")[0])
print("pip install ...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "diffusers", "transformers", "accelerate", "bitsandbytes",
    "peft", "trl", "cairosvg", "pillow", "tqdm", "sentencepiece"], check=True)

In [ ]:
# ── STEP 1: Write helper files ────────────────────────────────────────────
_VEC_B64 = "..."
_DS_B64 = "..."

for _fname, _b64 in [("vectorize.py", _VEC_B64), ("dataset_pipeline.py", _DS_B64)]:
    with open(_fname, "w", encoding="utf-8") as _f:
        _f.write(base64.b64decode(_b64).decode("utf-8"))
    print(f"Written: {_fname}")

# ── STEP 2: GPU check ─────────────────────────────────────────────────────
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Runtime -> Change runtime type -> T4 GPU")
print(f"GPU: {torch.cuda.get_device_name(0)}  ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")
os.environ["HF_TOKEN"] = HF_TOKEN

# ── STEP 3: Get results.json (Colab + Kaggle compatible) ─────────────────
if not Path(RESULTS_JSON).exists():
    import glob as _glob
    _kaggle_hits = _glob.glob("/kaggle/input/*/results.json")
    if _kaggle_hits:
        import shutil as _shutil
        _shutil.copy(_kaggle_hits[0], RESULTS_JSON)
        print(f"Copied from Kaggle dataset: {_kaggle_hits[0]}")
    else:
        try:
            from google.colab import files as _colab_files
            print("Upload results.json ...")
            uploaded = _colab_files.upload()
            for name in uploaded:
                Path(name).rename(RESULTS_JSON)
            print(f"Saved: {RESULTS_JSON}")
        except ImportError:
            raise FileNotFoundError(
                "results.json not found. "
                "On Kaggle: click Add Data -> Upload -> add results.json as a dataset."
            )
else:
    print(f"Found: {RESULTS_JSON}")

# ── GPU sanity check + auto-cleanup ──────────────────────────────────────
_free_gb = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved(0)) / 1e9
if _free_gb < 4.0:
    print(f"GPU only {_free_gb:.1f} GB free — attempting auto-cleanup of stale models ...")
    for _stale in ["ft_model", "trainer", "pipeline", "generate_image", "qwen_model", "qwen_processor"]:
        try:
            del globals()[_stale]
            print(f"  deleted {_stale}")
        except KeyError:
            pass
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    _free_gb = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved(0)) / 1e9
    print(f"  GPU free after cleanup: {_free_gb:.1f} GB")
    if _free_gb < 4.0:
        raise RuntimeError(
            f"Still only {_free_gb:.1f} GB GPU free after cleanup.\n"
            "Fix: Runtime → Disconnect and delete runtime → Reconnect, then re-run."
        )
print(f"GPU free: {_free_gb:.1f} GB  ✓")

# ── Skip-generation flag (reuse existing dataset after a Runtime reset) ───
_train_jsonl = Path(OUTPUT_DIR) / "dataset_train.jsonl"

# FORCE_REGEN: clear existing dataset so pipeline re-runs with VLM filter
if FORCE_REGEN and _train_jsonl.exists():
    import shutil as _shutil; _shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
    Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
    print("FORCE_REGEN: cleared dataset — will re-generate with VLM filter.")

_SKIP_GENERATION = (not FORCE_REGEN) and _train_jsonl.exists() and _train_jsonl.stat().st_size > 100
if _SKIP_GENERATION:
    print()
    print(f"Found existing dataset: {_train_jsonl}  ({_train_jsonl.stat().st_size:,} bytes)")
    print("Skipping generation (set FORCE_REGEN=True to rebuild).")
    jsonl_path = Path(OUTPUT_DIR) / "dataset.jsonl"
    train_path = _train_jsonl
    val_path = None
else:  # _SKIP_GENERATION is False — run the full pipeline

    # ── STEP 4: Mine bad prompts ──────────────────────────────────────────
    from dataset_pipeline import mine_bad_prompts

    bad_prompts, breakdown = mine_bad_prompts(RESULTS_JSON, CLIP_THRESHOLD, DINO_THRESHOLD)
    print(f"Bad prompts: {breakdown['bad_total']}/{breakdown['total']}  "
          f"(failures={breakdown['failed']}, low_clip={breakdown['low_clip']}, low_dino={breakdown['low_dino']})")
    for i, p in enumerate(bad_prompts, 1): print(f"  {i:2d}. {p}")

    # ── STEP 5: Load Qwen2VL (only when USE_VLM_FILTER=True) ─────────────
    if USE_VLM_FILTER:
        # Load on CPU first to avoid occupying GPU during SD3.5-M generation
        from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
        print("\nLoading Qwen2VL (4-bit, starts on CPU) ...")
        _quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")
        qwen_model = Qwen2VLForConditionalGeneration.from_pretrained(
            VLM_MODEL, quantization_config=_quant, device_map="cpu", torch_dtype=None)
        qwen_processor = AutoProcessor.from_pretrained(VLM_MODEL)
        print("Qwen2VL loaded on CPU.")
    else:
        qwen_model = qwen_processor = None
        print("\nSkipping Qwen2VL load (USE_VLM_FILTER=False).")

    # ── STEP 6: Load SD 3.5 Medium (takes GPU now that Qwen is on CPU) ───
    from dataset_pipeline import load_sd35
    generate_image = load_sd35(HF_TOKEN, simplification="high")
    print("SD 3.5 Medium loaded.")

    # ── STEP 7: Run dataset pipeline ──────────────────────────────────────
    logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", force=True)  # force=True overrides Jupyter's pre-existing handlers
    from dataset_pipeline import DatasetPipeline

    pipeline = DatasetPipeline(
        output_dir=OUTPUT_DIR, hf_token=HF_TOKEN, simplification="high",
        use_colour_vectorizer=False, vectorizer_threshold=0.45, vectorizer_resolution=512,
        diffusion_steps=SD_STEPS, diffusion_guidance=SD_GUIDANCE,
        save_images=True, seed=SD_SEED,
        use_vlm_filter=USE_VLM_FILTER,
        vlm_model=qwen_model, vlm_processor=qwen_processor, vlm_model_name=VLM_MODEL,
    )
    pipeline._generate_image = generate_image
    jsonl_path = pipeline.run(bad_prompts)
    print(f"Dataset: {jsonl_path}")
    # Print per-stage failure breakdown for debugging
    _stats_path = Path(OUTPUT_DIR) / "stats.json"
    if _stats_path.exists():
        print("Pipeline stats:", json.loads(_stats_path.read_text()))

    # ── STEP 8: Show samples ───────────────────────────────────────────────
    from IPython.display import SVG, display
    with open(jsonl_path, encoding="utf-8") as _f:
        records = [json.loads(l) for l in _f if l.strip()]
    print(f"\nDataset records: {len(records)}")
    for rec in records[:3]:
        print(f"Prompt: {rec['text']}")
        display(SVG(rec["svg"]))

    # ── STEP 9: Split dataset ─────────────────────────────────────────────
    train_path, val_path = DatasetPipeline.split_dataset(str(jsonl_path), train_ratio=0.9)
    print(f"Train: {train_path}  Val: {val_path}")

# ── STEP 10: Free ALL models before fine-tuning ───────────────────────────
print("\nFreeing all models from GPU before fine-tuning ...")
try:
    del qwen_model, qwen_processor
except: pass
try:
    del pipeline, generate_image
except: pass
# Also destroy the SD pipeline internals
import diffusers
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM free: {torch.cuda.memory_reserved(0)/1e9:.2f} GB reserved, "
      f"{torch.cuda.memory_allocated(0)/1e9:.2f} GB allocated")

# ── STEP 11: Fine-tune (fully inline) ────────────────────────────────────
from transformers import AutoTokenizer, TrainingArguments, Trainer, Qwen2VLForConditionalGeneration, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

SYSTEM_PROMPT = (
    "You are an expert SVG code generator. "
    "Given a text description, output ONLY valid SVG code. "
    "Rules:\n"
    '- Start with: <svg viewBox="0 0 200 200" xmlns="http://www.w3.org/2000/svg">\n'
    '- Use simple shapes: <rect>, <circle>, <ellipse>, <polygon>, <path>\n'
    '- Use solid hex fill colours (e.g., fill="#FF0000")\n'
    "- Keep it minimal: 5-20 elements\n- End with: </svg>\n"
    "Output the SVG code directly, no explanation."
)

def _build_chat(prompt, svg, tok):
    return tok.apply_chat_template([
        {"role":"system",    "content":SYSTEM_PROMPT},
        {"role":"user",      "content":f"Generate SVG for: {prompt}"},
        {"role":"assistant", "content":svg},
    ], tokenize=False, add_generation_prompt=False)

class _SVGDataset(torch.utils.data.Dataset):
    def __init__(self, recs, tok, ml=512): self.recs, self.tok, self.ml = recs, tok, ml
    def __len__(self): return len(self.recs)
    def __getitem__(self, idx):
        r = self.recs[idx]
        enc = self.tok(_build_chat(r["text"], r["svg"], self.tok),
                       truncation=True, max_length=self.ml, padding=False, return_tensors=None)
        ids = enc["input_ids"]; labs = list(ids)
        asst = self.tok.encode("<|im_start|>assistant", add_special_tokens=False)
        n, end = len(asst), 0
        for i in range(len(ids)-n):
            if ids[i:i+n] == asst: end = i+n
        for i in range(end): labs[i] = -100
        return {"input_ids":ids, "labels":labs, "attention_mask":enc["attention_mask"]}

class _Collator:
    def __init__(self, tok, ml=512): self.tok, self.ml = tok, ml
    def __call__(self, fs):
        ml = min(max(len(f["input_ids"]) for f in fs), self.ml); pad = self.tok.pad_token_id or 0
        return {
            "input_ids":      torch.tensor([(f["input_ids"]     +[pad] *(ml-len(f["input_ids"])))[:ml] for f in fs], dtype=torch.long),
            "labels":         torch.tensor([(f["labels"]        +[-100]*(ml-len(f["labels"])))[:ml]    for f in fs], dtype=torch.long),
            "attention_mask": torch.tensor([(f["attention_mask"]+[0]   *(ml-len(f["attention_mask"])))[:ml] for f in fs], dtype=torch.long),
        }

_train_recs = [json.loads(l) for l in open(str(train_path), encoding="utf-8") if l.strip()]
if not _train_recs:
    print("WARNING: dataset is empty — all diffusion steps failed. Cannot fine-tune.")
    print("This usually means Qwen2VL was not properly offloaded to CPU before SD generation.")
    print("Re-run after a Runtime -> Disconnect and delete runtime -> reconnect.")
else:
    _val_recs = [json.loads(l) for l in open(str(val_path), encoding="utf-8") if l.strip()] if val_path and Path(str(val_path)).exists() else []
    print(f"\nLoading model for fine-tuning ... Train: {len(_train_recs)}  Val: {len(_val_recs)}")

    tokenizer = AutoTokenizer.from_pretrained(VLM_MODEL, trust_remote_code=True, padding_side="right")
    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                                bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)
    ft_model = Qwen2VLForConditionalGeneration.from_pretrained(
        VLM_MODEL, quantization_config=quant, device_map="auto", trust_remote_code=True)
    ft_model = prepare_model_for_kbit_training(ft_model)
    ft_model = get_peft_model(ft_model, LoraConfig(
        task_type=TaskType.CAUSAL_LM, r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05,
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"], bias="none"))
    ft_model.print_trainable_parameters()

    trainer = Trainer(
        model=ft_model, processing_class=tokenizer,
        data_collator=_Collator(tokenizer),
        train_dataset=_SVGDataset(_train_recs, tokenizer),
        eval_dataset =_SVGDataset(_val_recs,   tokenizer) if _val_recs else None,
        args=TrainingArguments(
            output_dir=LORA_OUTPUT_DIR, num_train_epochs=EPOCHS,
            per_device_train_batch_size=BATCH_SIZE, gradient_accumulation_steps=GRAD_ACCUM,
            learning_rate=LEARNING_RATE, lr_scheduler_type="cosine", warmup_ratio=0.05,
            fp16=True, logging_steps=10,
            eval_strategy="steps" if _val_recs else "no", eval_steps=100 if _val_recs else None,
            save_strategy="steps", save_steps=200, save_total_limit=3,
            load_best_model_at_end=bool(_val_recs),
            metric_for_best_model="eval_loss" if _val_recs else None,
            report_to="none", seed=42, remove_unused_columns=False, dataloader_pin_memory=False,
        ),
    )
    print("Training ..."); trainer.train()

    adapter_path = os.path.join(LORA_OUTPUT_DIR, "final_adapter")
    ft_model.save_pretrained(adapter_path); tokenizer.save_pretrained(adapter_path)
    print(f"\nAdapter saved: {adapter_path}")

    import shutil as _shutil
    zip_path = _shutil.make_archive("qwen2vl_svg_lora", "zip", adapter_path)
    print(f"Adapter zipped: {zip_path}")
    try:
        from google.colab import files as _colab_files
        _colab_files.download(zip_path)
    except ImportError:
        print("On Kaggle: download via the Output tab (right panel).")
    print("Done!")


In [ ]:
# ── JSONL Dataset Filter: keep only simple SVGs ──────────────────────────────
import json, re, random
from pathlib import Path

MAX_PATHS   = 80    # keep SVGs with <= this many <path> elements
MAX_SVG_LEN = 6000  # skip very long SVGs (too many tokens)

_train_src = str(Path(OUTPUT_DIR) / "dataset_train.jsonl")
_val_src   = str(Path(OUTPUT_DIR) / "dataset_val.jsonl")

if not Path(_train_src).exists():
    print("No dataset found — run dataset generation first.")
else:
    # Load ALL records from both splits, filter, then re-split
    all_records = []
    for path in [_train_src, _val_src]:
        if Path(path).exists():
            all_records += [json.loads(l) for l in Path(path).read_text().splitlines() if l.strip()]

    before = len(all_records)
    kept = []
    for r in all_records:
        svg = r.get("svg", "")
        n = len(re.findall(r"<path", svg))
        if 1 <= n <= MAX_PATHS and len(svg) <= MAX_SVG_LEN:
            kept.append(r)
    print(f"Filter: {before} -> {len(kept)} records (removed {before-len(kept)})")

    # Re-split 90/10
    random.seed(42)
    random.shuffle(kept)
    split = max(1, int(len(kept) * 0.1))
    val_recs, train_recs = kept[:split], kept[split:]
    Path(_train_src).write_text(chr(10).join(json.dumps(r) for r in train_recs))
    Path(_val_src).write_text(chr(10).join(json.dumps(r) for r in val_recs))
    print(f"Train: {len(train_recs)}  Val: {len(val_recs)}")